# Lab 2: a small RAG, in Google Colab
**TsunamIA | Introduction to LLMs**

Runs in your browser on Google servers (no local install, no firewall on the download).

**How to use:** run cells **1, 2, 3 once** (3 loads the model, about 1-2 min). Then just edit and re-run **cell 4** (retrieval) and **cell 5** (full answer) as many times as you want; the model stays loaded.

Tip: Runtime > Change runtime type > **T4 GPU** for speed.

## 1. Setup: embed a small knowledge base *(run once)*

In [ ]:
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

docs = [
    "A PNR (Passenger Name Record) stores a traveler itinerary, dates and contact details.",
    "Refund rules depend on the fare: flexible fares allow free changes, basic economy usually does not.",
    "Baggage allowance varies by airline and fare class; extra bags are charged separately.",
    "Check-in opens 24 hours before departure and closes about 1 hour before for most carriers.",
    "Loyalty points are earned per flight and can be redeemed for upgrades or free tickets.",
    "Seat selection is free for some fares and paid for others, depending on the airline.",
    "An embedding turns text into a vector of numbers that captures its meaning.",
    "RAG retrieves the most relevant passages and gives them to the model to answer.",
]
doc_emb = embedder.encode(docs, normalize_embeddings=True)
print("Indexed", len(docs), "passages.")

## 2. Retrieval function *(run once)*

In [ ]:
def search(query, top_k=3):
    scores = doc_emb @ embedder.encode([query], normalize_embeddings=True)[0]
    order = np.argsort(-scores)[:top_k]
    print(f"Query: {query}\nTOP_K = {top_k}\n")
    for i in order:
        print(f"  {scores[i]:.3f}  {docs[i]}")
    return [docs[i] for i in order]

print("search() ready.")

## 3. Load the model + answer function *(run once, about 1-2 min)*

In [ ]:
from transformers import pipeline
gen = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def answer(query, top_k=3):
    ctx = "\n".join(f"- {d}" for d in search(query, top_k))
    msg = [{"role":"user","content":
        f"Answer the question using ONLY the context.\n\nContext:\n{ctx}\n\nQuestion: {query}"}]
    out = gen(msg, max_new_tokens=120, do_sample=False)[0]["generated_text"]
    print("\n--- ANSWER ---\n" + out[-1]["content"].strip())

print("answer() ready. Edit cells 4 and 5 below.")

## 4. ▶️ Experiment: RETRIEVAL
Change the two lines below and re-run (Shift+Enter). Try TOP_K = 1, then 3, then 8.

In [ ]:
# ===== EDIT THESE TWO LINES =====
QUERY = "Can I get my money back if I cancel?"
TOP_K = 3
# ================================
search(QUERY, TOP_K)

## 5. ▶️ Experiment: FULL ANSWER (RAG)
Same idea. The model is already loaded, so this is fast. Compare the answer with TOP_K = 1, 3, 8.

In [ ]:
# ===== EDIT THESE TWO LINES =====
QUERY = "Can I get a refund if I cancel my flight?"
TOP_K = 3
# ================================
answer(QUERY, TOP_K)